In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("../") 
if str(PROJECT_ROOT.resolve()) not in sys.path:
    sys.path.append(str(PROJECT_ROOT.resolve()))

In [9]:
import cv2
import yaml
from datetime import datetime
from tqdm import tqdm
from models.model_loader import load_model, get_device
from config import *
import numpy as np

In [10]:
def load_class_names(yaml_path,model):
    class_map = {}
    if not yaml_path:
        print(f"Warning: data.yaml not found at '{yaml_path}'. Class names will be default IDs.")
        return model.names
    try:
        with open(yaml_path, 'r', encoding='utf-8') as f:
            data = yaml.safe_load(f)
        if 'names' in data and isinstance(data['names'], list):
            class_map = {i: str(name) for i, name in enumerate(data['names'])}
            print(f"Loaded {len(class_map)} class names from '{yaml_path.name}'.")
        return class_map
    except Exception as e:
        print(f"Error loading class names from '{yaml_path}': {e}")
        return {}

def annotate_image_with_results(original_image, results, class_names_map):
    annotated_image = original_image.copy()
    
    if not results or not hasattr(results[0], 'boxes'):
        return annotated_image

    for box in results[0].boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        conf = float(box.conf[0])
        cls_id = int(box.cls[0])
        class_name = class_names_map.get(cls_id, f"ID_{cls_id}")
        
        color = (0, 255, 0)
        label = f"{class_name} {conf:.2f}"
        
        cv2.rectangle(annotated_image, (x1, y1), (x2, y2), color, 2)
        (w, h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(annotated_image, (x1, y1 - h - 5), (x1 + w, y1), color, -1)
        cv2.putText(annotated_image, label, (x1, y1 - 3), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)
        
    return annotated_image

In [ ]:
IMAGE_DIR = Path(r"C:\Users\swest\Desktop\Test Images")
OUTPUT_DIR = Path(f"./yolo_benchmarking_output/{datetime.now().isoformat().replace(':','')}")
MODEL_PATH = Path(r"C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\models\weights\20072025\few_anchowy_win_fp16.engine")
DATA_YAML_PATH = ""

MODEL_INPUT_SIZE = 640
CONF_THRESHOLD = 0.01
IOU_THRESHOLD = 0.6

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def run_benchmarking():
    device = get_device()
    model = load_model(str(MODEL_PATH), device, CONF_THRESHOLD, IOU_THRESHOLD)
    class_names = load_class_names(DATA_YAML_PATH, model)

    image_extensions = ['.jpg', '.jpeg', '.png', '.bmp']
    image_paths = [p for p in IMAGE_DIR.glob("**/*") if p.suffix.lower() in image_extensions]

    if not image_paths:
        print(f"No images found in '{IMAGE_DIR}'. Please check the path.")
        return

    print(f"\nFound {len(image_paths)} images. Starting inference...")

    for image_path in tqdm(image_paths, desc="Annotating Images"):
        try:
            frame = cv2.imread(str(image_path))
            if frame is None:
                print(f"Warning: Could not read image {image_path.name}. Skipping.")
                continue
            
            frame = cv2.resize(frame, (MODEL_INPUT_SIZE, MODEL_INPUT_SIZE), interpolation=cv2.INTER_LINEAR)
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
            # Step 1: Glare Reduction (Highlight Suppression)
            # Convert to grayscale to easily find the brightest spots
            gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
            
            # Create a mask of the overexposed regions (headlights)
            # The threshold 225 is a good starting point, can be tuned
            glare_mask = cv2.threshold(gray, 225, 255, cv2.THRESH_BINARY)[1]
            
            # Reduce the intensity of the glare regions in the original color image
            # We use a dampening factor (e.g., 0.7) to reduce brightness without making it black
            frame[glare_mask == 255] = (frame[glare_mask == 255] * 0.7).astype(np.uint8)
            
            # Step 2: Low-Light Enhancement (CLAHE on LAB color space)
            # Convert to LAB color space to apply CLAHE only on the Lightness channel
            lab = cv2.cvtColor(frame, cv2.COLOR_RGB2LAB)
            l, a, b = cv2.split(lab)
            
            # Apply CLAHE to the L-channel
            l_clahe = clahe.apply(l)
            
            # Merge the CLAHE-enhanced L-channel back with original A and B channels
            lab_clahe = cv2.merge((l_clahe, a, b))
            
            # Convert back to RGB color space
            frame = cv2.cvtColor(lab_clahe, cv2.COLOR_LAB2RGB)


            # Use the loaded model for prediction
            results = model.predict(frame, verbose=False, augment=True, half=True, rect=True, visualize=True)
            
            annotated_frame = annotate_image_with_results(frame, results, class_names)

            output_path = OUTPUT_DIR / image_path.name
            cv2.imwrite(str(output_path), annotated_frame)

        except Exception as e:
            print(f"An error occurred while processing {image_path.name}: {e}")

    print("\nBenchmarking complete.")
    print(f"Annotated images have been saved to: {OUTPUT_DIR.resolve()}")

# Execute the process
run_benchmarking()

2025-09-08 01:39:04.923 | DEBUG    | models.model_loader:load_model:11 - Loading model from: C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\models\weights\20072025\few_anchowy_win_fp16.engine onto device: cuda
2025-09-08 01:39:04.924 | DEBUG    | models.model_loader:load_model:50 - Detected .engine file, using TensorRT runtime via YOLO wrapper.
2025-09-08 01:39:04.925 | DEBUG    | models.model_loader:load_model:53 - TensorRT engine loaded successfully from: C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\models\weights\20072025\few_anchowy_win_fp16.engine


Using GPU: NVIDIA GeForce RTX 3090
Loading C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\models\weights\20072025\few_anchowy_win_fp16.engine for TensorRT inference...

Found 100 images. Starting inference...


Annotating Images: 100%|██████████| 100/100 [00:06<00:00, 16.23it/s]


Benchmarking complete.
Annotated images have been saved to: C:\Users\swest\Desktop\traffic-capturing\traffic-reporting\utility_scripts\yolo_benchmarking_output\2025-09-08T013904.922591


: 